# 04 · Hashrate Derivatives & Hashprice Model

**Purpose:** Model hashrate as a tradeable exposure. Hashprice is the fundamental unit of mining economics:

```
Hashprice ($/PH/day) = (block_subsidy + avg_fees) × blocks_per_day × BTC_price
                       / network_hashrate_PH
```

**Key derivatives:**
- **CME CF Bitcoin Hash Rate Index (BRRR)**: regulated hashrate futures
- **Luxor Hashrate Futures**: OTC hashrate forwards
- **Mining equity**: public miners are implicitly long hashrate exposure

**Data sources:**
- CoinMetrics: hashrate, difficulty, fees (notebook 01 data)
- Binance: BTC spot price
- `src/models/hashprice.py`: hashprice formula and profitability models

---

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from src.data.coinmetrics import fetch_hashrate_difficulty
from src.data.yfinance_fetcher import fetch_prices
from src.models.hashprice import (
    hashprice_usd, hashprice_series, mining_profit_per_ph_day,
    breakeven_hashprice, breakeven_btc_price, profitability_matrix,
    difficulty_ribbon
)
from src.utils.plotting import plot_hashprice, PLOTLY_TEMPLATE, BTC_ORANGE
from config import (
    DEFAULT_LOOKBACK_DAYS, CURRENT_BLOCK_SUBSIDY, BLOCKS_PER_DAY,
    ASIC_MODELS, ELECTRICITY_COSTS
)

pd.options.display.float_format = '{:,.4f}'.format
print('Setup complete.')

## 1. Load On-Chain Data & BTC Price

In [ ]:
onchain = fetch_hashrate_difficulty(days=DEFAULT_LOOKBACK_DAYS)
btc_prices = fetch_prices(['BTC-USD'], period='2y')['BTC-USD']

hashrate_ehs = onchain['hashrate']   # EH/s
difficulty   = onchain['difficulty']
blocks_per_day = onchain['blocks']
fees_btc_total = onchain['fees_btc']
fees_per_block = (fees_btc_total / blocks_per_day).rename('fees_per_block_btc')

# Align BTC price to on-chain data dates
btc_aligned = btc_prices.reindex(hashrate_ehs.index, method='ffill')

print(f'Date range:  {hashrate_ehs.index[0].date()} → {hashrate_ehs.index[-1].date()}')
print(f'Hashrate:    {hashrate_ehs.iloc[-1]:.1f} EH/s')
print(f'BTC price:   ${btc_aligned.iloc[-1]:,.0f}')
print(f'Fees/block:  {fees_per_block.iloc[-1]:.4f} BTC')

## 2. Compute Hashprice Time Series

In [ ]:
hp_series = hashprice_series(
    btc_price=btc_aligned,
    hashrate_ehs=hashrate_ehs,
    block_subsidy=CURRENT_BLOCK_SUBSIDY,
    fees_btc=fees_per_block,
)

current_hp = hp_series.iloc[-1]
print(f'Current hashprice: ${current_hp:.2f}/PH/day')
print(f'52w high:          ${hp_series.iloc[-365:].max():.2f}/PH/day')
print(f'52w low:           ${hp_series.iloc[-365:].min():.2f}/PH/day')
print(f'52w avg:           ${hp_series.iloc[-365:].mean():.2f}/PH/day')

In [ ]:
fig = plot_hashprice(hp_series, title='Bitcoin Hashprice ($/PH/day)')

# Add April 2024 halving marker
fig.add_vline(
    x='2024-04-19', line_dash='dot', line_color='#9B59B6',
    annotation_text='Halving (Apr 2024)', annotation_position='top right',
)
# Add 90-day moving average
hp_90ma = hp_series.rolling(90).mean()
fig.add_trace(go.Scatter(
    x=hp_90ma.index, y=hp_90ma.values,
    name='90d MA', line=dict(color='white', width=2, dash='dot'),
))
fig.show()

## 3. Hashprice Decomposition

Hashprice has three drivers: BTC price, network hashrate (difficulty), and transaction fees.
We decompose the change in hashprice into each component.

In [ ]:
# Subsidy-only hashprice (no fees)
hp_subsidy_only = hashprice_series(
    btc_aligned, hashrate_ehs,
    block_subsidy=CURRENT_BLOCK_SUBSIDY, fees_btc=0.0,
)

# Fee contribution
hp_fees = hp_series - hp_subsidy_only

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=hp_subsidy_only.index, y=hp_subsidy_only.values,
    name='Subsidy Revenue', stackgroup='one',
    fill='tozeroy', line=dict(width=0), fillcolor='rgba(247,147,26,0.6)',
))
fig.add_trace(go.Scatter(
    x=hp_fees.index, y=hp_fees.values,
    name='Fee Revenue', stackgroup='one',
    fill='tonexty', line=dict(width=0), fillcolor='rgba(52,152,219,0.6)',
))
fig.update_layout(
    title='Hashprice Decomposition: Subsidy vs Transaction Fees',
    yaxis_title='$/PH/day', xaxis_title='Date',
    template=PLOTLY_TEMPLATE, height=450,
)
fig.show()

fee_share = (hp_fees / hp_series).mean() * 100
print(f'Average fee share of hashprice: {fee_share:.1f}%')

## 4. Mining Profitability by ASIC Model & Electricity Cost

In [ ]:
# Breakeven hashprice for each ASIC at each electricity cost
print(f'{'ASIC Model':<25} {'Efficiency':>12}', end='')
for elec in ELECTRICITY_COSTS:
    print(f'  ${elec:.2f}/kWh', end='')
print()
print('-' * 85)

for asic, eff in ASIC_MODELS.items():
    print(f'{asic:<25} {eff:>10.1f} J/TH', end='')
    for elec in ELECTRICITY_COSTS:
        be = breakeven_hashprice(eff, elec)
        status = '✓' if current_hp > be else '✗'
        print(f'  ${be:>6.1f} {status}', end='')
    print()

print(f'\nCurrent hashprice: ${current_hp:.2f}/PH/day')
print('✓ = profitable at current hashprice  ✗ = underwater')

In [ ]:
# Net profit per PH/day at current hashprice
hp_levels = [current_hp * f for f in [0.5, 0.75, 1.0, 1.25, 1.5, 2.0]]
pm = profitability_matrix(hp_levels, ASIC_MODELS, ELECTRICITY_COSTS)

# Show profitability at current hashprice
current_pm = profitability_matrix([current_hp], ASIC_MODELS, ELECTRICITY_COSTS)
print(f'\nNet profit ($/PH/day) at current hashprice ${current_hp:.2f}:')
print(current_pm.round(2).to_string())

In [ ]:
# Heatmap: profitability matrix
asic_list = list(ASIC_MODELS.keys())
elec_list = ELECTRICITY_COSTS
z_matrix = [[mining_profit_per_ph_day(current_hp, ASIC_MODELS[a], e) for a in asic_list] for e in elec_list]

fig = go.Figure(go.Heatmap(
    z=z_matrix,
    x=asic_list,
    y=[f'${e:.2f}/kWh' for e in elec_list],
    colorscale='RdYlGn',
    text=[[f'${v:.0f}' for v in row] for row in z_matrix],
    texttemplate='%{text}',
    colorbar=dict(title='$/PH/day'),
))
fig.update_layout(
    title=f'Mining Profitability ($/PH/day) at Hashprice ${current_hp:.2f}',
    xaxis_title='ASIC Model', yaxis_title='Electricity Cost',
    template=PLOTLY_TEMPLATE, height=350,
)
fig.show()

## 5. Break-even BTC Price Curves

In [ ]:
current_hashrate = hashrate_ehs.iloc[-1]

fig = go.Figure()
colors = ['#F7931A', '#E74C3C', '#27AE60', '#3498DB', '#9B59B6']

for (asic, eff), color in zip(ASIC_MODELS.items(), colors):
    be_prices = []
    elec_range = np.linspace(0.03, 0.15, 30)
    for elec in elec_range:
        be = breakeven_btc_price(
            current_hashrate, eff, elec,
            block_subsidy=CURRENT_BLOCK_SUBSIDY,
        )
        be_prices.append(be)

    fig.add_trace(go.Scatter(
        x=elec_range, y=be_prices,
        name=asic, mode='lines',
        line=dict(color=color, width=2),
    ))

# Mark current BTC price
btc_now = btc_aligned.iloc[-1]
fig.add_hline(y=btc_now, line_dash='dash', line_color='white',
               annotation_text=f'Current BTC ${btc_now:,.0f}',
               annotation_position='top right')

fig.update_layout(
    title=f'Break-even BTC Price by ASIC Model & Electricity Cost\n(Hashrate: {current_hashrate:.0f} EH/s)',
    xaxis_title='Electricity Cost ($/kWh)',
    yaxis_title='Break-even BTC Price ($)',
    template=PLOTLY_TEMPLATE, height=500,
)
fig.show()

## 6. Hashprice vs BTC Price Sensitivity

In [ ]:
# Hashprice sensitivity: how much does hashprice change per $1k BTC move?
btc_range = np.linspace(30_000, 200_000, 100)
hp_at_btc = [hashprice_usd(p, current_hashrate, CURRENT_BLOCK_SUBSIDY) for p in btc_range]
hp_at_btc_highfee = [hashprice_usd(p, current_hashrate, CURRENT_BLOCK_SUBSIDY, avg_fees_per_block_btc=0.5) for p in btc_range]

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=btc_range, y=hp_at_btc,
    name='Normal fees (~0.1 BTC/block)', line=dict(color=BTC_ORANGE, width=2),
))
fig.add_trace(go.Scatter(
    x=btc_range, y=hp_at_btc_highfee,
    name='High fees (~0.5 BTC/block)', line=dict(color='#27AE60', width=2, dash='dot'),
))

# Mark current BTC price
fig.add_vline(x=btc_aligned.iloc[-1], line_dash='dash', line_color='white',
               annotation_text=f'Current BTC', annotation_position='top right')
fig.add_hline(y=current_hp, line_dash='dot', line_color='gray',
               annotation_text=f'Current HP ${current_hp:.0f}', annotation_position='right')

fig.update_layout(
    title=f'Hashprice Sensitivity to BTC Price (Hashrate={current_hashrate:.0f} EH/s)',
    xaxis_title='BTC Price ($)',
    yaxis_title='Hashprice ($/PH/day)',
    template=PLOTLY_TEMPLATE, height=450,
)
fig.show()

## 7. Post-Halving Impact Analysis

The April 2024 halving reduced the block subsidy from 6.25 → 3.125 BTC, cutting the subsidy component of hashprice by 50%. The network absorbed this via BTC price appreciation and increased fees.

In [ ]:
halving_date = '2024-04-19'

# Compare hashprice before and after halving with counterfactual
pre_halving = hp_series[hp_series.index < halving_date].tail(180)
post_halving = hp_series[hp_series.index >= halving_date]

# What would hashprice be post-halving with OLD subsidy (6.25 BTC)?
hp_counter = hashprice_series(
    btc_aligned[btc_aligned.index >= halving_date],
    hashrate_ehs[hashrate_ehs.index >= halving_date],
    block_subsidy=6.25,  # pre-halving subsidy
    fees_btc=fees_per_block[fees_per_block.index >= halving_date],
)

fig = go.Figure()
fig.add_trace(go.Scatter(x=pre_halving.index, y=pre_halving.values,
                          name='Pre-Halving HP', line=dict(color='#3498DB', width=2)))
fig.add_trace(go.Scatter(x=post_halving.index, y=post_halving.values,
                          name='Post-Halving HP (actual)', line=dict(color=BTC_ORANGE, width=2)))
fig.add_trace(go.Scatter(x=hp_counter.index, y=hp_counter.values,
                          name='Post-Halving HP (if subsidy=6.25)', 
                          line=dict(color='#27AE60', width=2, dash='dot')))
fig.add_vline(x=halving_date, line_dash='dot', line_color='#9B59B6',
               annotation_text='Halving', annotation_position='top right')
fig.update_layout(
    title='Post-Halving Hashprice Impact (Actual vs Counterfactual)',
    yaxis_title='$/PH/day', xaxis_title='Date',
    template=PLOTLY_TEMPLATE, height=450,
)
fig.show()

if len(post_halving) > 0:
    avg_actual = post_halving.mean()
    avg_counter = hp_counter.mean()
    print(f'Post-halving avg HP (actual):      ${avg_actual:.2f}/PH/day')
    print(f'Post-halving avg HP (old subsidy): ${avg_counter:.2f}/PH/day')
    print(f'Difference (BTC price appreciation offset): ${avg_counter - avg_actual:.2f}/PH/day')

## 8. Hashrate Derivatives Overview

### CME CF Bitcoin Hash Rate Index (BRRR)
The CME launched hash rate futures in 2022, settled against the CF Bitcoin Hash Rate Reference Rate.
These allow miners to hedge revenue uncertainty and investors to gain hashrate exposure.

### Luxor Hashrate Futures
Luxor Technology offers OTC hashrate forwards for miners to lock in future hashprice.

### Synthetic Hashrate Exposure
Mining company equity provides implicit hashrate exposure — modeled in notebook 05.

In [ ]:
# Summary table
print('Hashrate Derivatives Summary')
print('=' * 60)
products = [
    ('CME Hash Rate Futures', 'BRRR', 'Monthly', 'Cash (USD)', 'CF BHRR'),
    ('Luxor Hashrate Forwards', 'OTC', 'Custom', 'Cash (USD)', 'Luxor HP Index'),
    ('Mining Equity (e.g. MARA)', 'MARA', 'N/A', 'Stock', 'Implicit'),
    ('BTC Perpetual (proxy)', 'BTCUSDT-PERP', 'Perpetual', 'USDT', 'BTC price'),
]
print(f'{'Product':<30} {'Symbol':<15} {'Tenor':<12} {'Settlement':<12} {'Reference'}')
print('-' * 80)
for p in products:
    print(f'{p[0]:<30} {p[1]:<15} {p[2]:<12} {p[3]:<12} {p[4]}')

print(f'\nCurrent Hashprice: ${current_hp:.4f}/PH/day  (= ${current_hp*1e6/1e6:.4f}/TH/day)')
print(f'Annualized Hashrate Revenue: ${current_hp * 365:,.0f}/PH/year')

## Summary

- **Hashprice** is BTC price ÷ hashrate × block economics — the fundamental KPI for miners
- The 2024 halving cut subsidy revenue by 50%; network survived via price appreciation
- Most-efficient ASICs (S21 Pro at 15 J/TH) remain profitable at $0.07/kWh down to ~$X hashprice
- Hashrate derivatives allow miners to hedge and investors to gain isolated hashrate beta

**Next:** [05 · Public Mining Companies →](./05_public_mining_companies.ipynb)